# When during the 14 days does the gap appear?

**Data used:** the experiment itself (who got a CN reply + Views/Likes/Shares)

**Short answer:** Nowhere in particular — and that’s the finding. CN’d tweets run above Control by about the same amount every single day.



# Engagement Velocity / Trajectory Analysis

**Papers:**
- *Question / framing:* Vosoughi, Roy, Aral 2018, *The spread of true and false news online* (Science). Asks whether the *temporal pattern* of spread differs between groups, not just final magnitude.
- *Methodology / metrics:* Cheng, Adamic, Dow, Kleinberg, Leskovec 2014, *Can cascades be predicted?* (WWW). Defines temporal-shape feature taxonomy: first-half rate, second-half rate, time-to-*k*, slope of inter-arrival times.

**Goal.** Test whether CN replies change the *shape* of the 14-day engagement trajectory — not just the day-13 total. Directly addresses the views-vs-likes puzzle: if Treatment tweets accumulate engagement faster (or saturate sooner), that's a temporal mechanism distinct from "more total engagement."

**Six shape parameters per (tweet, metric):**

| # | Metric | Definition | Cheng 2014 anchor |
|---|---|---|---|
| 1 | First-window share | cum_d1 / cum_d13 | `time'_{1..k/2}` first-half rate |
| 2 | Half-life day | first day where cumulative ≥ 50% of final | `time_i` time-to-*k* |
| 3 | Late-window share | (cum_d13 − cum_d6) / cum_d13 | `time'_{k/2..k}` second-half rate (Cheng's most predictive) |
| 4 | Mean accumulation day | Σ t·Δ_t / Σ Δ_t | `time'_{1..k}` overall timing distribution |
| 5 | Acceleration index | late_window_share / first_window_share | `time''_{1..k}` slope of inter-arrival |
| 6 | Plateau day | first day where Δ_t < 5% of final | inspired extension — saturation timing |

All parameters computed on **post-baseline cumulative growth** (cum_t = Day_t − Day_0), matching the growth-DV convention from prior notebooks. Three metrics: Views, Likes, Shares (Comments excluded — too sparse).


## Section 0 — Config & Imports

In [ ]:
import os
from pathlib import Path

BASE_DIR = Path.cwd()
while not (BASE_DIR / 'data').is_dir() and BASE_DIR != BASE_DIR.parent:
    BASE_DIR = BASE_DIR.parent
DATA_DIR  = BASE_DIR / 'data'
OUT_DIR   = BASE_DIR / 'outputs' / '04_trajectory_shape'
OUT_DIR.mkdir(parents=True, exist_ok=True)

CONTROL_XL    = DATA_DIR / 'Control_Group.xlsx'
TREATMENT_XL  = DATA_DIR / 'Treatment_Group.xlsx'
MONITORING_XL = DATA_DIR / 'Tweet Monitoring.xlsx'

METRICS      = ['Views', 'Likes', 'Shares']
DAYS         = list(range(0, 14))      # day 0 .. day 13 inclusive
MAIN_WINDOW  = 13
N_BOOTSTRAP  = 2000
RANDOM_SEED  = 42

print(f'Base dir: {BASE_DIR}')
print(f'Out  dir: {OUT_DIR}')


In [ ]:
import json, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('ignore')
np.random.seed(RANDOM_SEED)
print('Imports OK')


In [ ]:
# Domain-aware date parser (Dec=2025, Jan=2026)
# Disambiguates DD/MM vs MM/DD using the experiment window: Dec=2025, Jan=2026
def parse_mixed_date(date_val):
    if pd.isna(date_val): return pd.NaT
    date_str = str(date_val).strip()
    def is_valid(year, month):
        return (year == 2025 and month == 12) or (year == 2026 and month == 1)
    if '-' in date_str and date_str[:4].isdigit():
        parts = date_str.split('-')
        year = int(parts[0]); num1 = int(parts[1]); num2 = int(parts[2].split()[0])
        if is_valid(year, num1):   month, day = num1, num2
        elif is_valid(year, num2): month, day = num2, num1
        else:                       month, day = num1, num2
        return pd.Timestamp(year=year, month=month, day=day)
    elif '/' in date_str:
        parts = date_str.split('/')
        num1 = int(parts[0]); num2 = int(parts[1]); year = int(parts[2].split()[0])
        if is_valid(year, num2):   day, month = num1, num2
        elif is_valid(year, num1): month, day = num1, num2
        else:                       day, month = num1, num2
        return pd.Timestamp(year=year, month=month, day=day)
    else:
        return pd.to_datetime(date_val, errors='coerce')

print('Date parser ready.')


## Section 1 — Load Data & Build Per-Tweet Trajectories

Pivot `Tweet Monitoring.xlsx` into a `(URL × day × metric)` matrix. Domain-aware date parser. Forward-fill the rare missing-day cell within each URL, then derive **post-baseline cumulative growth** `cum_t = Day_t − Day_0`.


In [ ]:
# Load raw data
control    = pd.read_excel(CONTROL_XL)
treatment  = pd.read_excel(TREATMENT_XL)
monitoring = pd.read_excel(MONITORING_XL)

print(f'Control:    {control.shape}')
print(f'Treatment:  {treatment.shape}')
print(f'Monitoring: {monitoring.shape}')

# Domain-aware date parsing — gives correct Dec 2025 / Jan 2026 dates
monitoring['Sample_Date']           = monitoring['Sample_Date'].apply(parse_mixed_date)
monitoring['Start_Date (Creation)'] = monitoring['Start_Date (Creation)'].apply(parse_mixed_date)
monitoring['Day'] = (monitoring['Sample_Date'] - monitoring['Start_Date (Creation)']).dt.days

# Sanity check
print(f"\nSample_Date  range: {monitoring['Sample_Date'].min().date()} → {monitoring['Sample_Date'].max().date()}")
print(f"Creation     range: {monitoring['Start_Date (Creation)'].min().date()} → {monitoring['Start_Date (Creation)'].max().date()}")
print(f"Day offsets seen:  min={monitoring['Day'].min()}, max={monitoring['Day'].max()}")


In [ ]:
# Pivot to (URL × Day) × metric matrix
pivoted = monitoring.pivot_table(
    index='URL', columns='Day',
    values=METRICS, aggfunc='first'
)
pivoted.columns = [f'{m}_Day{d}' for m, d in pivoted.columns]
pivoted = pivoted.reset_index()

# Attach group label
group_map = pd.concat([
    control[['URL']].assign(Group='Control'),
    treatment[['URL']].assign(Group='Treatment'),
])
df = pivoted.merge(group_map, on='URL', how='left')
df['is_treatment'] = (df['Group'] == 'Treatment').astype(int)

print(f'Pivoted: {df.shape}')
print(f"Group counts: {df['Group'].value_counts().to_dict()}")

# Per-day coverage
print('\nPer-day non-null coverage (%):')
for m in METRICS:
    cols = [f'{m}_Day{d}' for d in DAYS if f'{m}_Day{d}' in df.columns]
    pct = df[cols].notna().mean() * 100
    print(f'  {m}: ' + ' '.join(f'd{d}={p:5.1f}' for d, p in zip(DAYS, pct)))


In [ ]:
# Forward-fill within URL across days for each metric (rare missing cells)
# Cumulative metrics are monotone-ish in expectation; ffill is safe.
for m in METRICS:
    cols = [f'{m}_Day{d}' for d in DAYS if f'{m}_Day{d}' in df.columns]
    df[cols] = df[cols].ffill(axis=1)
    # Day0 NaN with no prior value → fill with 0 (matches convention from prior tasks)
    if f'{m}_Day0' in df.columns:
        df[f'{m}_Day0'] = df[f'{m}_Day0'].fillna(0)
        df[cols] = df[cols].ffill(axis=1)

# Build a (URL, day) cumulative-growth tensor: cum_t = Day_t - Day_0
# Stored as separate per-metric DataFrames, indexed by URL, columns = days 0..13
cum_growth = {}
for m in METRICS:
    cols = [f'{m}_Day{d}' for d in DAYS]
    if not all(c in df.columns for c in cols):
        continue
    raw = df.set_index('URL')[cols]
    raw.columns = DAYS
    # Cumulative *growth* relative to Day 0
    cg = raw.subtract(raw[0], axis=0)
    cum_growth[m] = cg

for m, cg in cum_growth.items():
    final_pos = (cg[MAIN_WINDOW] > 0).sum()
    final_zero = (cg[MAIN_WINDOW] == 0).sum()
    final_neg = (cg[MAIN_WINDOW] < 0).sum()
    print(f'{m}: final cum_growth — {final_pos} positive, {final_zero} zero, {final_neg} negative ({len(cg)} total)')


## Section 2 — Compute Six Shape Parameters Per (Tweet, Metric)

All shape parameters are computed on the post-baseline cumulative growth `cum_t = Day_t − Day_0`. A tweet is dropped from a metric's analysis if `cum_d13 ≤ 0` (no growth → all shape parameters undefined).

| # | Parameter | Formula |
|---|---|---|
| 1 | first_window_share | cum_d1 / cum_d13 |
| 2 | half_life_day | min{t : cum_t ≥ 0.5 × cum_d13} |
| 3 | late_window_share | (cum_d13 − cum_d6) / cum_d13 |
| 4 | mean_accum_day | Σ_{t=1..13} t·Δ_t / Σ Δ_t  where Δ_t = cum_t − cum_{t−1} |
| 5 | accel_index | late_window_share / first_window_share (capped via log) |
| 6 | plateau_day | min{t : Δ_t < 0.05 × cum_d13} (= 13 if never reached) |


In [ ]:
def compute_shape_params(cg: pd.DataFrame) -> pd.DataFrame:
    """
    cg: rows = URL, cols = days 0..13, values = cumulative growth (Day_t - Day_0).
    Returns a per-URL DataFrame with the 6 shape parameters.
    """
    out = pd.DataFrame(index=cg.index)
    final = cg[MAIN_WINDOW]

    # (1) first-window share: cum_d1 / cum_d13
    out['first_window_share'] = cg[1] / final.replace(0, np.nan)

    # (2) half-life day
    half = 0.5 * final
    def _half_life(row):
        f = row[MAIN_WINDOW]
        if pd.isna(f) or f <= 0: return np.nan
        thr = 0.5 * f
        for t in DAYS:
            if row[t] >= thr:
                return t
        return MAIN_WINDOW
    out['half_life_day'] = cg.apply(_half_life, axis=1)

    # (3) late-window share: (cum_d13 - cum_d6) / cum_d13
    out['late_window_share'] = (final - cg[6]) / final.replace(0, np.nan)

    # (4) mean accumulation day = Σ t·Δ_t / Σ Δ_t  for t in 1..13
    deltas = cg.diff(axis=1).iloc[:, 1:]            # Δ_t for t=1..13
    deltas.columns = list(range(1, MAIN_WINDOW + 1))
    weights = deltas.clip(lower=0)                   # ignore rare negative deltas (deletes/etc.)
    weight_sum = weights.sum(axis=1)
    weighted_t = (weights * np.array(weights.columns)).sum(axis=1)
    out['mean_accum_day'] = np.where(weight_sum > 0, weighted_t / weight_sum, np.nan)

    # (5) acceleration index = late_share / first_share. Use log-ratio for symmetry/cap.
    fs = out['first_window_share'].replace(0, np.nan)
    out['accel_index'] = np.log((out['late_window_share'] + 1e-6) / (fs + 1e-6))

    # (6) plateau day: first day where Δ_t < 5% of final. Else 13.
    def _plateau(deltas_row, f):
        if pd.isna(f) or f <= 0: return np.nan
        thr = 0.05 * f
        for t, val in deltas_row.items():
            if val < thr:
                return t
        return MAIN_WINDOW
    out['plateau_day'] = [
        _plateau(deltas.loc[idx], final.loc[idx]) for idx in cg.index
    ]

    # Mark drop set (final ≤ 0) so caller knows N kept
    out['_keep'] = final > 0
    return out


shape_dfs = {}
for m, cg in cum_growth.items():
    sp = compute_shape_params(cg)
    sp = sp.merge(df[['URL', 'Group', 'is_treatment']].set_index('URL'),
                  left_index=True, right_index=True, how='left')
    shape_dfs[m] = sp
    kept = sp['_keep'].sum()
    total = len(sp)
    by_group = sp[sp['_keep']].groupby('Group').size().to_dict()
    print(f'{m}: {kept}/{total} tweets kept (cum_d13 > 0). By group: {by_group}')


In [ ]:
# Quick descriptive summary of shape parameters by metric and group
SHAPE_PARAMS = ['first_window_share', 'half_life_day', 'late_window_share',
                'mean_accum_day', 'accel_index', 'plateau_day']

for m in METRICS:
    sp = shape_dfs[m]
    keep = sp[sp['_keep']]
    print(f'\n=== {m}  (N={len(keep)}) ===')
    summary = keep.groupby('Group')[SHAPE_PARAMS].agg(['median', 'mean']).round(3)
    print(summary.to_string())


## Section 3 — Treatment vs. Control on Each Shape Parameter

For each (metric × shape parameter) — 3 × 6 = **18 contrasts**:

1. **Mann–Whitney U** with bootstrap CI on median difference (Treatment − Control).
2. **Regression** with baseline control: `shape_param ~ is_treatment + log_baseline` where `log_baseline = log1p(Day_0)` for that metric.
3. **BH-FDR** across all 18 contrasts (controls FDR at α = 0.05).

A negative MWU `r` (rank-biserial) means Treatment values are *lower*; positive means *higher*.


In [ ]:
def mwu_with_bootstrap(t_vals, c_vals, n_boot=N_BOOTSTRAP, seed=RANDOM_SEED):
    t_vals = np.asarray(t_vals, dtype=float); t_vals = t_vals[~np.isnan(t_vals)]
    c_vals = np.asarray(c_vals, dtype=float); c_vals = c_vals[~np.isnan(c_vals)]
    if len(t_vals) < 5 or len(c_vals) < 5:
        return dict(mwu_p=np.nan, mwu_r=np.nan, median_diff=np.nan, ci_lo=np.nan, ci_hi=np.nan,
                    n_t=len(t_vals), n_c=len(c_vals))
    u, p = stats.mannwhitneyu(t_vals, c_vals, alternative='two-sided')
    # rank-biserial (effect size). Negative = T < C.
    r = 1 - (2 * u) / (len(t_vals) * len(c_vals))
    median_diff = float(np.median(t_vals) - np.median(c_vals))
    rng = np.random.default_rng(seed)
    boots = np.empty(n_boot)
    for b in range(n_boot):
        ts = rng.choice(t_vals, size=len(t_vals), replace=True)
        cs = rng.choice(c_vals, size=len(c_vals), replace=True)
        boots[b] = np.median(ts) - np.median(cs)
    ci_lo, ci_hi = np.percentile(boots, [2.5, 97.5])
    return dict(mwu_p=float(p), mwu_r=float(r), median_diff=median_diff,
                ci_lo=float(ci_lo), ci_hi=float(ci_hi),
                n_t=len(t_vals), n_c=len(c_vals))


def regression_baseline_controlled(sp_df, param, baseline_log_col):
    sub = sp_df.dropna(subset=[param, baseline_log_col]).copy()
    sub['param'] = sub[param]
    formula = f'param ~ is_treatment + {baseline_log_col}'
    try:
        m = smf.ols(formula, data=sub).fit()
        return dict(reg_coef=float(m.params['is_treatment']),
                    reg_se=float(m.bse['is_treatment']),
                    reg_p=float(m.pvalues['is_treatment']),
                    reg_n=int(m.nobs))
    except Exception as e:
        return dict(reg_coef=np.nan, reg_se=np.nan, reg_p=np.nan, reg_n=0)


# Build the 18-row results table
results = []
for m in METRICS:
    sp = shape_dfs[m].copy()
    keep = sp[sp['_keep']].copy()
    # Attach log-baseline (Day_0 of the same metric)
    keep[f'log_baseline_{m}'] = np.log1p(df.set_index('URL').loc[keep.index, f'{m}_Day0'].values)
    for param in SHAPE_PARAMS:
        t_vals = keep.loc[keep['is_treatment'] == 1, param].values
        c_vals = keep.loc[keep['is_treatment'] == 0, param].values
        mwu = mwu_with_bootstrap(t_vals, c_vals)
        reg = regression_baseline_controlled(keep, param, f'log_baseline_{m}')
        results.append(dict(metric=m, param=param, **mwu, **reg))

res_df = pd.DataFrame(results)

# BH-FDR adjustment across all 18 (use regression p-values as primary; also adjust MWU p)
res_df['mwu_p_fdr'] = multipletests(res_df['mwu_p'].fillna(1.0), method='fdr_bh')[1]
res_df['reg_p_fdr'] = multipletests(res_df['reg_p'].fillna(1.0), method='fdr_bh')[1]

def stars(p):
    if pd.isna(p): return ''
    if p < 0.001: return '***'
    if p < 0.01:  return '**'
    if p < 0.05:  return '*'
    if p < 0.10:  return '.'
    return ''
res_df['reg_sig'] = res_df['reg_p_fdr'].apply(stars)
res_df['mwu_sig'] = res_df['mwu_p_fdr'].apply(stars)

display_cols = ['metric','param','n_t','n_c','median_diff','ci_lo','ci_hi','mwu_r',
                'mwu_p','mwu_p_fdr','mwu_sig','reg_coef','reg_se','reg_p','reg_p_fdr','reg_sig']
print('\n=== Shape parameter tests (T vs C), 18 contrasts, BH-FDR ===')
print(res_df[display_cols].to_string(index=False))


In [ ]:
# Save the results table
res_path = OUT_DIR / 'shape_param_t_vs_c.csv'
res_df.to_csv(res_path, index=False)
print(f'Saved: {res_path}')

# Also save the shape parameters themselves (per-tweet, per-metric)
for m, sp in shape_dfs.items():
    out = sp[sp['_keep']].drop(columns=['_keep']).reset_index()
    out.to_csv(OUT_DIR / f'shape_params_{m}.csv', index=False)
    print(f'Saved: shape_params_{m}.csv  ({len(out)} rows)')


In [ ]:
# Forest plot — standardized effect (median_diff / pooled IQR) per (metric, param)
import matplotlib.patches as mpatches

fig, axes = plt.subplots(1, 3, figsize=(16, 6), sharey=True)

for ax, m in zip(axes, METRICS):
    sub = res_df[res_df['metric'] == m].copy().reset_index(drop=True)
    # standardize median_diff against the pooled IQR for that param's distribution
    keep = shape_dfs[m]
    keep = keep[keep['_keep']]
    iqrs = {}
    for p in SHAPE_PARAMS:
        v = keep[p].dropna().values
        if len(v) > 0:
            iqrs[p] = max(np.subtract(*np.percentile(v, [75, 25])), 1e-6)
        else:
            iqrs[p] = 1.0
    sub['std_diff'] = sub.apply(lambda r: r['median_diff'] / iqrs[r['param']], axis=1)
    sub['std_lo']   = sub.apply(lambda r: r['ci_lo']      / iqrs[r['param']], axis=1)
    sub['std_hi']   = sub.apply(lambda r: r['ci_hi']      / iqrs[r['param']], axis=1)

    y = np.arange(len(sub))
    colors = ['firebrick' if s else '0.4' for s in sub['reg_sig']]
    ax.errorbar(sub['std_diff'], y,
                xerr=[sub['std_diff'] - sub['std_lo'], sub['std_hi'] - sub['std_diff']],
                fmt='o', color='black', ecolor='gray', capsize=3, markersize=6)
    for yi, c in zip(y, colors):
        ax.plot(sub['std_diff'].iloc[yi], yi, 'o', color=c, markersize=8, zorder=3)
    ax.axvline(0, color='black', lw=0.8)
    ax.set_yticks(y)
    ax.set_yticklabels(sub['param'])
    ax.set_xlabel('Treatment − Control (median diff / pooled IQR)')
    ax.set_title(m)
    ax.grid(axis='x', alpha=0.3)

# legend
fig.legend(handles=[
    mpatches.Patch(color='firebrick', label='reg p_FDR < 0.05'),
    mpatches.Patch(color='0.4',       label='ns'),
], loc='lower center', ncol=2, bbox_to_anchor=(0.5, -0.02))

plt.suptitle('Treatment vs. Control on engagement-trajectory shape parameters', y=1.02)
plt.tight_layout()
plt.savefig(OUT_DIR / 'forest_shape_params.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: forest_shape_params.png')


## Section 4 — Mean Trajectory Plots (Linear + Log-y)

For each metric, plot the mean cumulative trajectory by group with bootstrap 95% CI band, on linear and log-y axes. The log-y view tends to amplify early-window differences that the linear view compresses.


In [ ]:
def bootstrap_mean_curves(cg: pd.DataFrame, group_labels: pd.Series,
                            n_boot=500, seed=RANDOM_SEED):
    rng = np.random.default_rng(seed)
    out = {}
    for grp in ['Control', 'Treatment']:
        idx = group_labels.index[group_labels == grp]
        sub = cg.loc[idx].dropna(how='all').values
        boots = np.empty((n_boot, sub.shape[1]))
        for b in range(n_boot):
            samp = sub[rng.integers(0, len(sub), size=len(sub))]
            boots[b] = np.nanmean(samp, axis=0)
        out[grp] = dict(
            mean=np.nanmean(sub, axis=0),
            lo=np.percentile(boots, 2.5, axis=0),
            hi=np.percentile(boots, 97.5, axis=0),
        )
    return out


fig, axes = plt.subplots(2, 3, figsize=(16, 8))
group_labels = df.set_index('URL')['Group']

for col, m in enumerate(METRICS):
    cg = cum_growth[m]
    curves = bootstrap_mean_curves(cg, group_labels)
    for row, scale in enumerate(['linear', 'log']):
        ax = axes[row, col]
        for grp, color in [('Control', 'tab:blue'), ('Treatment', 'tab:orange')]:
            c = curves[grp]
            ax.plot(DAYS, c['mean'], '-', color=color, label=grp, lw=2)
            ax.fill_between(DAYS, c['lo'], c['hi'], color=color, alpha=0.2)
        if scale == 'log':
            # Replace zeros with eps for log scale
            ax.set_yscale('symlog', linthresh=1)
        ax.set_xlabel('Day since detection')
        ax.set_ylabel(f'Cumulative {m} growth')
        ax.set_title(f'{m} — {scale} y')
        ax.grid(alpha=0.3)
        ax.legend()

plt.tight_layout()
plt.savefig(OUT_DIR / 'mean_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: mean_trajectories.png')


In [ ]:
# Visual sanity check: 50 random per-tweet trajectories per group, faceted
fig, axes = plt.subplots(2, 3, figsize=(16, 8), sharex=True)

rng = np.random.default_rng(RANDOM_SEED)
group_labels = df.set_index('URL')['Group']

for col, m in enumerate(METRICS):
    cg = cum_growth[m]
    for row, grp in enumerate(['Control', 'Treatment']):
        idx = group_labels.index[group_labels == grp]
        sample_idx = rng.choice(idx, size=min(50, len(idx)), replace=False)
        ax = axes[row, col]
        for u in sample_idx:
            ax.plot(DAYS, cg.loc[u].values, color='black', alpha=0.15, lw=0.7)
        ax.plot(DAYS, cg.loc[sample_idx].mean(axis=0).values,
                color='firebrick' if grp == 'Treatment' else 'navy', lw=2,
                label=f'{grp} mean (n=50 sample)')
        ax.set_yscale('symlog', linthresh=1)
        ax.set_title(f'{m} — {grp}')
        ax.set_xlabel('Day')
        ax.set_ylabel(f'Cum {m} growth')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)

plt.tight_layout()
plt.savefig(OUT_DIR / 'sample_trajectories.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: sample_trajectories.png')


## Section 5 — Trajectory Clustering (Exploratory)

Z-score-normalize each tweet's 14-day trajectory (so we cluster on **shape**, not magnitude). Run k-means on the normalized vectors. Then χ² test whether cluster composition differs between Treatment and Control.

Run for Views only (most data, clearest dynamics). K = 3 by default — small enough for interpretability, large enough to separate "very front-loaded" / "moderate" / "slow-growing" archetypes.


In [ ]:
K = 3
metric_for_clustering = 'Views'

cg = cum_growth[metric_for_clustering]
keep_mask = (cg[MAIN_WINDOW] > 0)
cg_keep = cg[keep_mask].copy()

# Normalize each tweet's trajectory: divide by its own day-13 final.
# This focuses on shape: each tweet's curve goes from 0 (day 0) to 1 (day 13).
shape_mat = cg_keep.div(cg_keep[MAIN_WINDOW], axis=0).fillna(0).values

km = KMeans(n_clusters=K, random_state=RANDOM_SEED, n_init=10)
clusters = km.fit_predict(shape_mat)

cluster_df = pd.DataFrame({
    'URL': cg_keep.index,
    'cluster': clusters,
}).merge(df[['URL', 'Group', 'is_treatment']], on='URL', how='left')

# Plot cluster centroids
fig, ax = plt.subplots(1, 1, figsize=(8, 5))
for c in range(K):
    centroid = km.cluster_centers_[c]
    n_in = (clusters == c).sum()
    ax.plot(DAYS, centroid, lw=2, label=f'Cluster {c} (n={n_in})')
ax.set_xlabel('Day')
ax.set_ylabel('Normalized cumulative growth (curve / day-13 final)')
ax.set_title(f'{metric_for_clustering} trajectory clusters (k={K})')
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.savefig(OUT_DIR / 'cluster_centroids.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: cluster_centroids.png')


In [ ]:
# Cluster × Group table + chi-square test
ct = pd.crosstab(cluster_df['cluster'], cluster_df['Group'])
print('Cluster × Group counts:')
print(ct)

# Row percentages
print('\nWithin-cluster % of Treatment:')
print((ct['Treatment'] / ct.sum(axis=1) * 100).round(1).to_string())

chi2, p, dof, expected = stats.chi2_contingency(ct)
print(f'\nChi-square: chi2={chi2:.3f}  dof={dof}  p={p:.4f}')

# Save
cluster_df.to_csv(OUT_DIR / 'cluster_assignments.csv', index=False)
ct.to_csv(OUT_DIR / 'cluster_x_group.csv')
print(f'\nSaved: cluster_assignments.csv, cluster_x_group.csv')


## Section 6 — Main Insights & Conclusion

### Headline finding

**CN replies do not detectably change the *shape* of engagement accumulation.** None of the 18 contrasts (3 metrics × 6 shape parameters) is significant after BH-FDR correction. Lowest FDR-adjusted p-value is **0.051** (Views × half-life day, MWU) — and even there the median difference is exactly 0.

*Source: `shape_param_t_vs_c.csv`. Every row in `reg_sig` and `mwu_sig` is blank.*

---

### Insight 1 — Engagement is heavily front-loaded for **all** tweets, regardless of group

About 80% of post-baseline growth happens on day 1, half-life is day 1, and tweets plateau by day 2 — for both groups, on every metric. This is a property of how X content propagates, not a treatment effect.

| Metric | First-window share (mean) C / T | Plateau day (mean) C / T | Half-life day (median) C / T |
|---|---|---|---|
| Views  | 78.7% / 81.4% | 2.53 / 2.61 | 1 / 1 |
| Likes  | 89.7% / 88.0% | 2.17 / 2.15 | 1 / 1 |
| Shares | 81.4% / 80.1% | 1.97 / 2.12 | 1 / 1 |

*Sources: descriptive group summary printed in Section 2; `shape_params_*.csv`. Visually obvious in `mean_trajectories.png` — the Treatment and Control mean curves overlap almost perfectly.*

---

### Insight 2 — The views-vs-likes puzzle has **no** temporal explanation

Going in, one plausible mechanism for "Treatment got more views but not more likes/shares" was: CN replies trigger an early algorithmic boost (more views in the first 24h), but the additional viewers don't engage. That story would predict **Treatment Views more front-loaded, shorter half-life, earlier plateau.**

We see none of it. On Views, all 6 shape parameters cluster around zero with overlapping CIs:

- First-window share: T 81.4% vs C 78.7% — diff +2.7 pp, FDR p = 0.55
- Half-life day: medians identical (1.0 / 1.0)
- Plateau day: medians identical (2.0 / 2.0)
- Late-window share: 4.3% / 4.6% — essentially equal

**Conclusion: the anti-suppression effect on Views must operate on magnitude, not on temporal dynamics.** The mechanism is "how many people see it," not "how quickly they see it."

*Source: `forest_shape_params.png` Views panel; `shape_param_t_vs_c.csv` Views rows.*

---

### Insight 3 — A few sub-threshold raw signals exist but disappear under FDR

For transparency, the contrasts with raw p < 0.10 (none survive correction; most have median diff = 0):

| Metric × Param | mwu_p_raw | reg_p_raw | mwu_p_FDR | What it would have meant |
|---|---|---|---|---|
| Views × half_life_day | **0.003** | 0.103 | 0.051 | Slight rank shift; medians identical |
| Views × first_window_share | 0.66 | **0.031** | 0.55 | T +2.6 pp more front-loaded — substantively negligible |
| Views × plateau_day | 0.16 | **0.098** | 0.55 | T plateaus +0.078 days later |
| Shares × accel_index | **0.016** | 0.24 | 0.15 | T Shares decelerate slightly faster |
| Shares × late_window_share | **0.048** | 0.66 | 0.29 | T slightly more late-window Shares |

These are the kind of pattern you expect from running 18 tests against a noisy null. Treat them as noise unless future analyses give them independent corroboration.

*Source: `shape_param_t_vs_c.csv`.*

---

### Insight 4 — Trajectory clustering also shows no group asymmetry

K-means (k=3) on shape-normalized Views growth curves yields three archetypes (n = 784 / 81 / 306). Treatment share within each: **50.8% / 40.7% / 51.3%**. Only the small middle cluster (n=81) shows any imbalance, and it's driven by 33 vs 48 raw counts — within sampling noise for a cluster that small.

*Source: `cluster_x_group.csv`, `cluster_centroids.png`.*

---

### Conclusion — what this analysis contributes to the paper

The trajectory analysis returns a **substantive null**: CN replies do not detectably alter the *temporal shape* of engagement accumulation, only the magnitude. This matters in two ways:

1. **It rules out a class of mechanism stories.** "CN replies attract early algorithmic boost," "CN replies cause earlier saturation," "CN replies stretch the engagement tail" — none are supported. Whatever drives Treatment Views > Control Views, it doesn't show up as a faster/slower or differently-shaped curve. It shows up as the **same curve at a higher level**.

2. **It strengthens the level-only mechanism story.** When we discuss the views-vs-likes asymmetry, we can write: *"the divergence cannot be explained by differential temporal dynamics — both groups concentrate engagement in the first 24 hours and saturate by day 2–3."*

**Caveats:**
- Engagement curves are extremely front-loaded (half-life day = 1 for almost every tweet), so shape-parameter variance is low and power to detect group differences is genuinely limited.
- Likes (N=446) and Shares (N=186) are conditional samples — only tweets with non-zero post-baseline growth are included. The "did the tweet grow at all" question is the main-effect analysis (already answered in `main_effect.ipynb`).
- This is a between-group test on the marginal shape. Shape differences *within specific moderator subgroups* (e.g., only for high-CN-toxicity cases) are not ruled out; that would be a follow-up.

---

### Outputs index

Saved to `outputs/trajectory_shape/`:

- `shape_param_t_vs_c.csv` — 18-row results table (MWU + regression, raw + FDR p-values).
- `shape_params_{Views,Likes,Shares}.csv` — per-tweet shape parameters (one row per included tweet).
- `forest_shape_params.png` — forest plot of standardized T−C effects across all 18 contrasts.
- `mean_trajectories.png` — mean ± bootstrap-95%-CI cumulative-growth curves by group, linear and log-y.
- `sample_trajectories.png` — 50 random per-tweet trajectories per group, faceted.
- `cluster_centroids.png` — k=3 k-means cluster centroids on Views shape-normalized trajectories.
- `cluster_assignments.csv`, `cluster_x_group.csv` — per-tweet cluster labels and the cluster × group contingency table.